# Tutorial: Custom Pulse Annotations and OpenQASM 3 Export

This tutorial covers:

1. Creating custom `PulseEnvelopeAnnotation` for specific gates
2. Attaching annotations to `BoxOp` blocks
3. Overriding default pulse shapes (DRAG, GaussianSquare parameters)
4. Exporting to OpenQASM 3 with serialized pulse pragmas
5. Round-trip serialization via QPY

In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.circuit.controlflow import BoxOp
from qiskit_waveform_sim import (
    TargetWaveformSimulator,
    PulseEnvelopeAnnotation,
    PulseAnnotationSerializer,
)
from qiskit import qasm3
import qiskit.qpy as qpy
import io

## 1. Understanding PulseEnvelopeAnnotation

In [2]:
# Create annotations with different pulse shapes

# Default DRAG (used for sx, x, rx)
drag_default = PulseEnvelopeAnnotation()
print(f"Default DRAG: {drag_default}")

# Custom high-beta DRAG for leakage suppression
drag_high_beta = PulseEnvelopeAnnotation(
    shape="drag",
    amp=0.55,           # Peak amplitude
    beta=0.25,          # DRAG beta (leakage suppression)
    sigma_ratio=0.2,    # σ as fraction of duration
    risefall_dt=16,     # Not used for DRAG
)
print(f"High-beta DRAG: {drag_high_beta}")

# GaussianSquare for flat-top pulses (CR, measure)
gs_custom = PulseEnvelopeAnnotation(
    shape="gaussian_square",
    amp=0.7,
    beta=0.0,           # Not used for GS
    sigma_ratio=0.0,    # Not used for GS
    risefall_dt=24,     # Rise/fall in dt
)
print(f"GaussianSquare: {gs_custom}")

Default DRAG: PulseEnvelopeAnnotation(shape='drag', amp=0.5, beta=0.08, sigma_ratio=0.25, risefall_dt=16)
High-beta DRAG: PulseEnvelopeAnnotation(shape='drag', amp=0.55, beta=0.25, sigma_ratio=0.2, risefall_dt=16)
GaussianSquare: PulseEnvelopeAnnotation(shape='gaussian_square', amp=0.7, beta=0.0, sigma_ratio=0.0, risefall_dt=24)


## 2. Attaching Annotations to BoxOp

In [3]:
backend = GenericBackendV2(num_qubits=2, seed=42)
backend.target.add_instruction(BoxOp, name="box")

qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)

# Override the SX gate inside this box with custom pulse
with qc.box(
    duration=160,
    unit="dt",
    annotations=[drag_high_beta]
):
    qc.sx(0)

qc.measure([0, 1], [0, 1])

# Verify annotation is attached
box_op = qc.data[1].operation  # Second instruction is the box
box_op = qc.data[2].operation  # Third instruction is the box; index 1 is the cx gate
print(f"BoxOp annotations: {box_op.annotations}")
print(f"BoxOp duration: {box_op.duration} {box_op.unit}")

BoxOp annotations: [PulseEnvelopeAnnotation(shape='drag', amp=0.55, beta=0.25, sigma_ratio=0.2, risefall_dt=16)]
BoxOp duration: 160 dt


## 3. Compile and Verify Custom Pulse Applied

In [4]:
pm = generate_preset_pass_manager(
    optimization_level=1,
    backend=backend,
    scheduling_method=None,  # or just omit scheduling_method
)

scheduled_qc = pm.run(qc)

sim = TargetWaveformSimulator(backend.target).compile_unscheduled(scheduled_qc)

# Find the SX events on drive channel 0
d0_events = sim.events_by_channel.get("d0", [])
sx_events = [e for e in d0_events if e.op_name == "sx"]

print(f"SX events on d0: {len(sx_events)}")
for e in sx_events:
    print(f"  start={e.start_dt} dt, amp={abs(e.amp):.3f}, beta={e.beta:.3f}, σ={e.sigma_dt:.1f} dt")

# The annotated SX should have custom parameters
custom_sx = [e for e in sx_events if np.isclose(abs(e.amp), 0.55, rtol=0.01)]
print(f"\nCustom SX found: {len(custom_sx)} event(s)")
if custom_sx:
    e = custom_sx[0]
    print(f"  Amplitude: {abs(e.amp):.3f} (expected 0.55)")
    print(f"  Beta: {e.beta:.3f} (expected 0.25)")
    print(f"  Sigma: {e.sigma_dt:.1f} dt (expected 32.0)")

SX events on d0: 2
  start=0 dt, amp=0.450, beta=0.080, σ=48.8 dt
  start=3367 dt, amp=0.550, beta=0.250, σ=39.0 dt

Custom SX found: 1 event(s)
  Amplitude: 0.550 (expected 0.55)
  Beta: 0.250 (expected 0.25)
  Sigma: 39.0 dt (expected 32.0)


## 4. Multiple Annotations and Fractional Gates

In [5]:
# You can attach different annotations to different BoxOps
qc2 = QuantumCircuit(2, 2)
qc2.h(0)

# Box 1: Custom DRAG for RX(π/4)
with qc2.box(
    duration=120,
    unit="dt",
    annotations=[PulseEnvelopeAnnotation(shape="drag", amp=0.4, beta=0.15, sigma_ratio=0.25)]
):
    qc2.rx(np.pi/4, 0)

# Box 2: Custom GaussianSquare for RZZ
with qc2.box(
    duration=300,
    unit="dt",
    annotations=[PulseEnvelopeAnnotation(shape="gaussian_square", amp=0.8, risefall_dt=30)]
):
    qc2.rzz(np.pi/2, 0, 1)

qc2.measure([0, 1], [0, 1])

pm = generate_preset_pass_manager(
    optimization_level=1, backend=backend, scheduling_method=None
)
scheduled_qc2 = pm.run(qc2)
sim2 = TargetWaveformSimulator(backend.target).compile_unscheduled(scheduled_qc2)

# Check both custom pulses
for ch in ["d0", "u(0,1)"]:
    events = sim2.events_by_channel.get(ch, [])
    for e in events:
        if e.duration_dt > 0:
            print(f"{ch}: {e.op_name} amp={abs(e.amp):.3f} shape={e.shape}")

d0: sx amp=0.450 shape=drag
d0: sx amp=0.400 shape=drag
d0: sx amp=0.400 shape=drag
u(0,1): cx_cr+ amp=0.650 shape=gaussian_square
u(0,1): cx_cr- amp=0.650 shape=gaussian_square
u(0,1): cx_cr+ amp=0.650 shape=gaussian_square
u(0,1): cx_cr- amp=0.650 shape=gaussian_square


## 5. Export to OpenQASM 3 with Pulse Pragmas

In [6]:
# Export with custom annotation serializer
qasm3_str = qasm3.dumps(
    scheduled_qc,
    annotation_handlers={"pulse_sim.envelope": PulseAnnotationSerializer()}
)

print(qasm3_str[:2000])

OPENQASM 3.0;
include "stdgates.inc";
bit[2] c;
rz(pi/2) $0;
sx $0;
rz(pi/2) $0;
cx $0, $1;
@pulse_sim.envelope {'shape': 'drag', 'amp': 0.55, 'beta': 0.25, 'sigma_ratio': 0.2, 'risefall_dt': 16}
box[160dt] {
  sx $0;
}
c[0] = measure $0;
c[1] = measure $1;



## 6. QPY Round-trip Serialization

In [7]:
# Serialize to QPY binary format
buf = io.BytesIO()
qpy.dump(scheduled_qc, buf, annotation_factories={"pulse_sim.envelope": PulseAnnotationSerializer().as_qpy()})

# Deserialize
buf.seek(0)
loaded_qc = qpy.load(buf, annotation_factories={"pulse_sim.envelope": PulseAnnotationSerializer().as_qpy()})[0]

# Verify annotations survived round-trip
for inst in loaded_qc.data:
    op = inst.operation
    if hasattr(op, 'annotations') and op.annotations:
        for ann in op.annotations:
            if isinstance(ann, PulseEnvelopeAnnotation):
                print(f"Restored annotation: {ann}")

Restored annotation: PulseEnvelopeAnnotation(shape='drag', amp=0.55, beta=0.25, sigma_ratio=0.2, risefall_dt=16)


## 7. Export to Hardware Formats (LabOne Q, RFSoC/QICK)

In [8]:
# Extract waveforms for hardware upload
# This is a simplified example - real hardware needs specific formats

# Get full circuit duration
total_dt = sim.total_duration_dt

# Extract all channels
waveforms = {}
for ch in sim.get_channels():
    snip = sim.get_snippet(ch, start_dt=0, length_dt=total_dt)
    waveforms[ch] = {
        "time_ns": snip.time_ns,
        "I": snip.wave.real,
        "Q": snip.wave.imag,
        "phase": snip.phase_rad,
        "dt": snip.dt_sec,
    }
    print(f"{ch}: {len(snip.wave)} samples")

# For LabOne Q: convert to their waveform format
# For RFSoC/QICK: convert to their DAC format (int16, etc.)
# See the hardware_export tutorial for complete examples

d0: 16171 samples
m0: 16171 samples
m1: 16171 samples
u(0,1): 16171 samples


## Summary

✅ **Custom pulses**: Override any gate's pulse shape via `BoxOp` + `PulseEnvelopeAnnotation`

✅ **Parametric control**: Amplitude, beta, sigma, risefall all configurable

✅ **OpenQASM 3 export**: Pulse annotations serialize as `@pulse_sim.drag(...)` pragmas

✅ **QPY support**: Full binary round-trip with annotations preserved

✅ **Hardware ready**: Extract I/Q arrays for LabOne Q, RFSoC, QICK, etc.

Next: See :doc:`hardware_export` for complete hardware integration examples.